# Wayu: สร้างเสียงให้ครบ pilot Train 80 / Dev 20

ขั้นนี้ใช้ **ข้อความของชุดเสียงคนที่แบ่งไว้แล้ว** สร้างเสียง Wayu 100 คลิป: Train 80 / Dev 20 ใช้ model/revision/voice/seed/speed เดิมจาก trial 10 คลิป ใช้เสียงเดิมที่ตรง provenance แทนการสร้างทับ จึงเพิ่มจริง 90 คลิปในเครื่องนี้

นี่เป็น **pilot เพื่อลองกระบวนการฝึก** ไม่ใช่ชุดวิจัยเต็ม ไม่ใช่ Final Test และยังไม่ train AASIST/RawNet2 หรือเติม noise/telephone ไม่ป้อนเสียงคนให้ TTS ไม่ใช่ voice cloning

เลือก **Kernel → .venv-wayu (Python 3.11)** แล้วรันจากบนลงล่าง ถ้าเครื่องใหม่ ให้มี raw Common Voice release เดียวกัน และรัน EDA ขั้นเตรียมทดลอง 1–3 เพื่อสร้าง `real_train.csv`, `real_dev.csv` และ `pilot_split_report.json` ก่อน จากนั้นรัน `.\scripts\setup_wayu_pilot.ps1 -Dataset` จาก project root

notebook นี้ไม่ต้องใช้ MMS หรือ manifest review10 ของ MMS น้ำหนัก/เสียง/ผลใน `outputs/` ไม่เข้า Git ดู [model card Wayu](https://huggingface.co/wayu-ai/wayu-paxa-tts-edge) และเงื่อนไข CC-BY-NC-4.0 / [ข้อกำหนดการใช้](https://www.wayuresearch.org/terms) ก่อนนำเสียงไปใช้หรือแจก

In [ ]:
from pathlib import Path
import sys, hashlib, json, io, time, platform, re
import importlib.metadata as metadata
import pandas as pd
import numpy as np
import soundfile as sf
from scipy.io import wavfile
from IPython.display import display, Audio, Markdown

PROJECT_ROOT = Path.cwd()
while not (PROJECT_ROOT / "pyproject.toml").is_file():
    if PROJECT_ROOT == PROJECT_ROOT.parent:
        raise FileNotFoundError("เปิด notebook ภายใน project ก่อน")
    PROJECT_ROOT = PROJECT_ROOT.parent
assert Path(sys.executable).resolve() == (PROJECT_ROOT / ".venv-wayu/Scripts/python.exe").resolve(), "เลือก .venv-wayu"
assert sys.version_info[:2] == (3, 11)

PILOT_DIR = PROJECT_ROOT / "data/processed/cvtts/pilot_v1"
MANIFEST_DIR = PILOT_DIR / "manifests"
QC_DIR = PILOT_DIR / "qc"
EXPERIMENT = "wayu_pilot_train_dev_v1"
split_report = json.loads((QC_DIR / "pilot_split_report.json").read_text(encoding="utf-8"))
real_tables, manifest_hashes = {}, {}
for split, size in [("train", 80), ("dev", 20)]:
    path = MANIFEST_DIR / f"real_{split}.csv"
    manifest_hashes[split] = hashlib.sha256(path.read_bytes()).hexdigest()
    assert manifest_hashes[split] == split_report["manifest_sha256"][path.name]
    frame = pd.read_csv(path, dtype="string", keep_default_na=False)
    assert len(frame) == size and frame["split"].eq(split).all()
    assert frame["label"].eq("bonafide").all() and frame["language"].eq("th").all()
    for column in ["sample_id", "client_id", "text_key", "audio_path", "audio_file_sha256", "pcm_sha256"]:
        assert frame[column].ne("").all() and frame[column].is_unique, column
    assert frame["sentence"].str.strip().ne("").all()
    real_tables[split] = frame.sort_values("sample_id").reset_index(drop=True)

for column in ["sample_id", "client_id", "text_key", "audio_path", "audio_file_sha256", "pcm_sha256"]:
    assert set(real_tables["train"][column]).isdisjoint(set(real_tables["dev"][column])), column

source_rows = [row.to_dict() for split in ["train", "dev"] for _, row in real_tables[split].iterrows()]
for row in source_rows:
    path = (PROJECT_ROOT / row["audio_path"]).resolve()
    assert path.is_relative_to((PROJECT_ROOT / "data/raw/common_voice").resolve())
    assert hashlib.sha256(path.read_bytes()).hexdigest() == row["audio_file_sha256"]
print("เสียงคนต้นทาง: Train 80 / Dev 20; hashes และ split ไม่ทับกัน")
display(pd.concat(real_tables.values(), ignore_index=True)[["sample_id", "sentence", "split"]].head())

## 1. โหลด Wayu ด้วยค่าที่กำหนดเดิม

ใช้ CPU, voice `m_young_clear`, seed 42 **รีเซ็ตต่อคลิป**, speed 1.0 ไม่สุ่ม voice ตามผลฟังหรือผล detector

มีหนึ่งข้อความที่มีคำอังกฤษ “Facebook” จึงต้องติดตั้ง `en_core_web_sm 3.8.0` ใน environment นี้ไว้ก่อน ไม่ปล่อยให้ frontend ดาวน์โหลด resource รุ่นใหม่ระหว่างรัน ดู [spaCy resource รุ่นที่ใช้](https://github.com/explosion/spacy-models/releases/tag/en_core_web_sm-3.8.0)

สร้างตาม native sample rate **24 kHz** ไม่บังคับ 4.04 วินาที ไม่ resample/crop/pad/normalize/เติม noise ในขั้นนี้

In [ ]:
import torch
import spacy
from huggingface_hub import snapshot_download
from wayu_tts import ThaiTTS
import wayu_tts

MODEL_ID = "wayu-ai/wayu-paxa-tts-edge"
MODEL_REVISION = "8196688df56a8d08ccfa186505343adff412ab6b"
CODE_REVISION = "07ab898d516649313926f0b843b809a962f993ad"
VOICE, SEED, SPEED = "m_young_clear", 42, 1.0
MODEL_DIR = PROJECT_ROOT / "checkpoints/tts/wayu-paxa-tts-edge" / MODEL_REVISION
MODEL_FILES = ["config.json", "model.pth", f"voices/{VOICE}.pt", "README.md"]
dist = metadata.distribution("wayu-tts")
direct_url = json.loads(dist.read_text("direct_url.json") or "{}")
assert CODE_REVISION in direct_url.get("url", "")
if any(re.search(r"[A-Za-z]", r["sentence"]) for r in source_rows):
    assert spacy.util.is_package("en_core_web_sm"), "รัน setup_wayu_pilot.ps1 -Dataset ก่อน"
    assert metadata.version("en-core-web-sm") == "3.8.0"
if not all((MODEL_DIR / name).is_file() for name in MODEL_FILES):
    snapshot_download(repo_id=MODEL_ID, revision=MODEL_REVISION,
                      local_dir=MODEL_DIR, allow_patterns=MODEL_FILES)

torch.set_num_threads(2)
torch.use_deterministic_algorithms(True)
tts = ThaiTTS.from_pretrained(MODEL_DIR, device="cpu")
sample_rate = int(tts.sample_rate)
assert sample_rate == 24000
model_hashes = {name: hashlib.sha256((MODEL_DIR / name).read_bytes()).hexdigest() for name in MODEL_FILES}
model_config = json.loads((MODEL_DIR / "config.json").read_text(encoding="utf-8"))
software = {p: metadata.version(p) for p in [
    "wayu-tts", "kokoro", "tltk", "pythainlp", "misaki", "torch", "transformers",
    "huggingface-hub", "numpy", "scipy", "scikit-learn", "gensim", "soundfile",
    "spacy", "en-core-web-sm",
]}
code_dir = Path(wayu_tts.__file__).parent
code_hashes = {p.name: hashlib.sha256(p.read_bytes()).hexdigest() for p in sorted(code_dir.glob("*.py"))}
print("Wayu:", MODEL_REVISION, "| voice:", VOICE, "| seed:", SEED, "| native:", sample_rate)

## 2. ตรวจ frontend ทั้ง 100 ข้อความก่อนสร้าง

ใช้ข้อความเดิม ไม่แก้สะกด ไม่เลือก seed จาก Dev ถ้า frontend รายงานว่ามีส่วนหายจะ **หยุด** และแสดงรายการปัญหา ไม่ตัดแถวทิ้งหรือประกาศว่าชุดครบแล้ว

trace ที่ผ่านไม่ได้ยืนยันว่าอ่านถูกทุกคำ การฟังผ่าน 10 คลิปแรกไม่ใช่การฟังผ่านอีก 90 คลิป ส่วน Dev ใช้ค่าตัวสร้างที่กำหนดแล้ว ไม่ใช้ผล detector หรือฟัง Dev เพื่อจูนตัวสร้างเสียง

In [ ]:
frontends = {}
frontend_failures = []
for row in source_rows:
    trace = tts.g2p.trace(row["sentence"])
    chunks = tts.phonemize(row["sentence"])
    if not trace.is_clean or not chunks or not all(tts.g2p.count_tokens(c) > 0 for c in chunks):
        frontend_failures.append({
            "sample_id": row["sample_id"], "split": row["split"], "text": row["sentence"],
            "dropped_words": trace.dropped_words, "dropped_symbols": sorted(trace.dropped_symbols),
            "phoneme_chunks": chunks,
        })
    frontends[row["sample_id"]] = (trace, chunks)
if frontend_failures:
    display(pd.DataFrame(frontend_failures))
    raise RuntimeError("พบ frontend ที่มีปัญหา ยังไม่สร้างชุดเต็ม ต้องตรวจสาเหตุก่อน")
assert len(frontends) == 100
print("frontend ตรวจครบ 100 ข้อความ ไม่พบส่วนหายที่ trace รายงาน")
print("ยังต้องฟังตรวจคำอ่านจริง ไม่ใช้ trace แทนคนฟัง")

## 3. สร้างเพิ่ม และใช้เสียงเดิมเมื่อที่มาตรงกัน

ตรวจเสียงเดิมกับ model/voice/seed/settings/ข้อความ/split/manifest/hash ก่อนใช้ ไม่เปลี่ยนรายงานเก่าหรือผลฟังของคน เสียงใหม่สร้างซ้ำใน memory เพื่อตรวจความตรงกัน พร้อมตรวจ finite/nonempty/nonzero และ WAV float32 roundtrip

เตรียมไฟล์ใหม่ใน memory แล้วตรวจทั้ง batch ก่อนเขียน ไม่มี timestamp หรือเวลาประมวลผลใน immutable provenance ถ้าพบ version เดิมต่างจะหยุด ไม่เขียนทับ

In [ ]:
fake_reports, pending_files, actions = [], {}, []
started = time.perf_counter()
for index, row in enumerate(source_rows, start=1):
    split = row["split"]
    sid = f"wayu_paxa_{row['sample_id']}_{VOICE}_seed{SEED}"
    wav_path = PILOT_DIR / "native_tts/wayu" / split / f"{sid}.wav"
    json_path = QC_DIR / "tts_samples" / f"{sid}.json"
    assert wav_path.exists() == json_path.exists(), f"คู่เสียง/รายงานไม่ครบ: {sid}"
    trace, chunks = frontends[row["sample_id"]]
    common = {
        "sample_id": sid, "audio_path": wav_path.relative_to(PROJECT_ROOT).as_posix(),
        "label": "spoof", "split": split, "language": "th",
        "source_sample_id": row["sample_id"], "source_text_key": row["text_key"],
        "source_audio_file_sha256": row["audio_file_sha256"], "tts_text": row["sentence"],
        "reference_audio_used": False,
        "generator_model": MODEL_ID, "generator_revision": MODEL_REVISION,
        "generator_code_repository": "https://github.com/wayu-research/wayu-tts-inference",
        "generator_code_revision": CODE_REVISION,
        "generator_license": "CC-BY-NC-4.0", "generator_code_license": "Apache-2.0",
        "voice": VOICE, "generation_seed": SEED, "device": "cpu", "cpu_threads": 2,
        "deterministic_algorithms": True,
        "settings": {"speed": SPEED, "calibrated_voice_speed": tts.config.voice_speed(VOICE)},
        "phoneme_chunks": chunks,
        "frontend_trace": {
            "dropped_words": trace.dropped_words, "dropped_symbols": sorted(trace.dropped_symbols),
            "is_clean": trace.is_clean, "not_a_human_pronunciation_verdict": True,
        },
        "model_config": model_config, "model_file_sha256": model_hashes,
        "sample_rate": sample_rate, "channels": 1, "wav_subtype": "FLOAT",
        "python": platform.python_version(),
    }
    if wav_path.exists():
        saved = json.loads(json_path.read_text(encoding="utf-8"))
        for key, value in common.items():
            assert saved.get(key) == value, f"ที่มาไฟล์เดิมไม่ตรง: {sid} / {key}"
        # รายงาน review10 รุ่นเดิมใช้ชื่อ source_train_manifest_sha256; ไม่แก้ schema เดิม
        if "source_manifest_sha256" in saved:
            assert saved["source_manifest_sha256"] == manifest_hashes[split]
            assert saved["source_manifest"] == (MANIFEST_DIR / f"real_{split}.csv").relative_to(PROJECT_ROOT).as_posix()
        else:
            assert split == "train" and saved["source_train_manifest_sha256"] == manifest_hashes[split]
        for package, version in saved["software"].items():
            assert metadata.version(package) == version, f"package เปลี่ยน: {package}"
        assert set(software) - {"spacy", "en-core-web-sm"} <= set(saved["software"])
        if "generator_code_file_sha256" in saved:
            assert saved["generator_code_file_sha256"] == code_hashes
        audio, native_sr = sf.read(wav_path, dtype="float32")
        assert native_sr == sample_rate and sf.info(wav_path).subtype == "FLOAT"
        wav_content = wav_path.read_bytes()
        action = "ใช้เดิม"
    else:
        audio = np.asarray(tts(row["sentence"], voice=VOICE, speed=SPEED, seed=SEED), dtype=np.float32)
        repeat = np.asarray(tts(row["sentence"], voice=VOICE, speed=SPEED, seed=SEED), dtype=np.float32)
        assert np.array_equal(audio, repeat), f"สร้างซ้ำไม่ตรง: {sid}"
        buffer = io.BytesIO()
        wavfile.write(buffer, sample_rate, audio)
        wav_content = buffer.getvalue()
        decoded, sr = sf.read(io.BytesIO(wav_content), dtype="float32")
        assert sr == sample_rate and np.array_equal(decoded, audio)
        action = "สร้างใหม่"
    assert audio.ndim == 1 and audio.size > 0 and np.isfinite(audio).all()
    assert float(np.abs(audio).max()) > 1e-8
    measurements = {
        "samples": int(audio.size), "duration_s": audio.size / sample_rate,
        "peak": float(np.abs(audio).max()),
        "audio_file_sha256": hashlib.sha256(wav_content).hexdigest(),
        "waveform_sha256": hashlib.sha256(audio.astype("<f4", copy=False).tobytes()).hexdigest(),
        "repeat_generation_waveform_equal": True,
    }
    if wav_path.exists():
        for key, value in measurements.items():
            assert saved.get(key) == value, f"ข้อมูลเสียงเดิมไม่ตรง: {sid} / {key}"
        item = saved
    else:
        item = {
            **common, **measurements, "experiment": EXPERIMENT,
            "source_manifest": (MANIFEST_DIR / f"real_{split}.csv").relative_to(PROJECT_ROOT).as_posix(),
            "source_manifest_sha256": manifest_hashes[split],
            "generator_code_file_sha256": code_hashes, "software": software,
            "manual_review": {"status": "pending", "text_matches": None, "notes": ""},
        }
        pending_files[wav_path] = wav_content
        pending_files[json_path] = (json.dumps(item, ensure_ascii=False, indent=2) + "\n").encode("utf-8")
    fake_reports.append(item)
    actions.append(action)
    print(f"{index:03d}/100 | {split:5s} | {action:8s} | {item['duration_s']:.2f}s | {row['sentence'][:55]}", flush=True)

assert len(fake_reports) == 100
assert len({r["audio_file_sha256"] for r in fake_reports}) == 100, "เสียงปลอมซ้ำ ต้องตรวจ ไม่บันทึกชุดเต็ม"
print("สร้าง/ตรวจครบ; ยังไม่เขียน batch จนกว่า manifests และ split จะผ่าน")

## 4. บันทึก manifest ของเสียงปลอม และรายการสองคลาส

บันทึก `tts_wayu_train.csv` (80), `tts_wayu_dev.csv` (20), `tts_wayu_all.csv` (100) และรายการสองคลาส `wayu_pilot_train_native.csv` (160), `wayu_pilot_dev_native.csv` (40)

**native** หมายถึงเสียงยังเป็น sample rate ต้นทาง: คนเป็น MP3 และ Wayu เป็น WAV 24 kHz รายการนี้ยังไม่ใช่คำสั่ง train ของ CLI เดิม และยังต้องจัดรูปแบบเสียงร่วมกันก่อนฝึก ไม่อ้างว่าปัญหา sample-rate/codec ระหว่างคลาสถูกควบคุมแล้ว

ตรวจ lineage และไม่ทับกันข้าม Train/Dev; รายงานเทคนิคไม่ใส่ความคิดเห็นส่วนตัวหรือประกาศว่าเสียงใหม่ทั้งชุดอ่านถูก

In [ ]:
fake_columns = [
    "sample_id", "source_sample_id", "audio_path", "tts_text", "source_text_key",
    "split", "label", "generator_model", "generator_revision", "voice",
    "sample_rate", "channels", "duration_s", "audio_file_sha256",
]
df_fake_all = pd.DataFrame([{k: r[k] for k in fake_columns} for r in fake_reports])
assert df_fake_all["sample_id"].is_unique and df_fake_all["source_sample_id"].is_unique
fake_tables = {s: df_fake_all[df_fake_all["split"].eq(s)].reset_index(drop=True) for s in ["train", "dev"]}
native_tables = {}
for split in ["train", "dev"]:
    assert set(fake_tables[split]["source_sample_id"]) == set(real_tables[split]["sample_id"])
    real_native = real_tables[split][[
        "sample_id", "audio_path", "split", "label", "sentence", "sample_rate",
        "channels", "duration_s", "audio_file_sha256",
    ]].copy()
    real_native["source_sample_id"] = real_tables[split]["sample_id"]
    real_native["source_text_key"] = real_tables[split]["text_key"]
    real_native["generator_model"], real_native["voice"] = "", ""
    fake_native = fake_tables[split].rename(columns={"tts_text": "sentence"})
    native_columns = [
        "sample_id", "audio_path", "split", "label", "source_sample_id", "sentence",
        "source_text_key", "generator_model", "voice", "sample_rate", "channels",
        "duration_s", "audio_file_sha256",
    ]
    native_tables[split] = pd.concat([real_native[native_columns], fake_native[native_columns]], ignore_index=True)
    assert native_tables[split]["sample_id"].is_unique
    assert native_tables[split]["audio_file_sha256"].is_unique, "เสียงจริง/ปลอมซ้ำใน split"

for column in ["source_sample_id", "source_text_key", "audio_file_sha256"]:
    assert set(native_tables["train"][column]).isdisjoint(set(native_tables["dev"][column])), column

manifest_tables = {
    "tts_wayu_train.csv": fake_tables["train"],
    "tts_wayu_dev.csv": fake_tables["dev"],
    "tts_wayu_all.csv": df_fake_all,
    "wayu_pilot_train_native.csv": native_tables["train"],
    "wayu_pilot_dev_native.csv": native_tables["dev"],
}
output_manifest_hashes = {}
for filename, frame in manifest_tables.items():
    content = frame.to_csv(index=False, lineterminator="\n", float_format="%.10g").encode("utf-8")
    pending_files[MANIFEST_DIR / filename] = content
    output_manifest_hashes[filename] = hashlib.sha256(content).hexdigest()

dataset_report = {
    "experiment": EXPERIMENT, "scope": "feasibility_pilot_not_final_research_evaluation",
    "source_release": split_report["source_release"],
    "source_manifest_sha256": manifest_hashes,
    "generator_model": MODEL_ID, "generator_revision": MODEL_REVISION,
    "generator_code_revision": CODE_REVISION, "generator_code_file_sha256": code_hashes,
    "model_file_sha256": model_hashes, "software": software,
    "voice": VOICE, "seed_per_clip": SEED, "speed": SPEED, "native_tts_sample_rate": sample_rate,
    "counts": {"train": {"bonafide": 80, "spoof": 80}, "dev": {"bonafide": 20, "spoof": 20}},
    "cross_split_overlap": {"source_sample_id": 0, "source_text_key": 0, "audio_file_sha256": 0},
    "manifest_sha256": output_manifest_hashes,
    "technical_checks": {
        "frontend_trace_clean": True, "finite_nonempty_nonzero": True,
        "float32_wav_roundtrip": True, "repeat_generation_verified_in_new_samples": True,
        "existing_samples_verified_against_provenance": True,
        "all_raw_audio_hashes_match": True,
    },
    "manual_review_source": "qc/tts_reviews/<sample_id>.json",
    "technical_checks_do_not_prove_all_words_correct": True,
    "no_resampling_cropping_noise_telephone_or_training": True,
    "native_manifests_not_yet_canonical_training_inputs": True,
}
REPORT_PATH = QC_DIR / "tts_experiments" / f"{EXPERIMENT}.json"
pending_files[REPORT_PATH] = (json.dumps(dataset_report, ensure_ascii=False, indent=2) + "\n").encode("utf-8")
for path, content in pending_files.items():
    if path.exists() and path.read_bytes() != content:
        raise FileExistsError(f"ไม่เขียนทับ version เดิมที่ต่าง: {path}")
for path, content in pending_files.items():
    path.parent.mkdir(parents=True, exist_ok=True)
    if not path.exists():
        with path.open("xb") as handle:
            handle.write(content)

for split in ["train", "dev"]:
    assert hashlib.sha256((MANIFEST_DIR / f"real_{split}.csv").read_bytes()).hexdigest() == manifest_hashes[split]
print("ครบ: เสียงคน 100 + Wayu 100; ไม่ทำสำเนาเสียงคน")
print("รอบนี้:", pd.Series(actions).value_counts().to_dict())
print("เวลา:", round(time.perf_counter() - started, 2), "วินาที")
display(pd.concat(native_tables.values(), ignore_index=True).groupby(["split", "label"]).size().rename("clips").to_frame())
display(df_fake_all.groupby("split")["duration_s"].agg(["min", "median", "max", "sum"]))

## 5. ดูสถานะและฟังตัวอย่างใหม่

โหลดผลฟังที่มีแล้วโดยไม่แก้ไฟล์ review ไม่ประกาศว่าอีก 90 คลิปผ่านการฟัง ตัวอย่างด้านล่างเลือก **Train** แบบกระจายความยาวและไม่ซ้ำคลิปที่เคยฟัง พร้อมให้ฟังข้อความที่มีคำอังกฤษ ถ้า Dev มีคำอ่านผิดให้บันทึกเพื่อ audit ไม่ปรับตัวสร้างตาม Dev ใน pilot รุ่นนี้

ขั้นถัดไปหลังตรวจตัวอย่าง: จัดเสียงทั้งสองคลาสเป็นรูปแบบร่วมกัน แล้วเพิ่ม training loop AASIST เพื่อทดลอง Clean 1 epoch ยังไม่ใช่ผลวิจัย

In [ ]:
review_rows = []
for item in fake_reports:
    review_path = QC_DIR / "tts_reviews" / f"{item['sample_id']}.json"
    status = "pending"
    if review_path.is_file():
        review = json.loads(review_path.read_text(encoding="utf-8"))
        assert review["sample_id"] == item["sample_id"]
        assert review["audio_file_sha256"] == item["audio_file_sha256"]
        status = review["status"]
    review_rows.append({"sample_id": item["sample_id"], "split": item["split"], "status": status})
df_reviews = pd.DataFrame(review_rows)
display(df_reviews.groupby(["split", "status"]).size().rename("clips").to_frame())

unheard_train = fake_tables["train"].merge(df_reviews[["sample_id", "status"]], on="sample_id")
unheard_train = unheard_train[unheard_train["status"].eq("pending")].sort_values(["duration_s", "sample_id"]).reset_index(drop=True)
if len(unheard_train):
    positions = sorted(set(np.linspace(0, len(unheard_train) - 1, min(5, len(unheard_train))).round().astype(int)))
    listen_rows = unheard_train.iloc[positions].copy()
    english_rows = unheard_train[unheard_train["tts_text"].str.contains(r"[A-Za-z]", regex=True)]
    listen_rows = pd.concat([listen_rows, english_rows], ignore_index=True).drop_duplicates("sample_id")
    for i, row in enumerate(listen_rows.to_dict("records"), start=1):
        path = (PROJECT_ROOT / row["audio_path"]).resolve()
        assert path.is_relative_to(PILOT_DIR.resolve())
        assert hashlib.sha256(path.read_bytes()).hexdigest() == row["audio_file_sha256"]
        display(Markdown(f"### ตัวอย่างใหม่ {i} — Train — {row['duration_s']:.2f} วินาที"))
        print("ข้อความ:", row["tts_text"])
        display(Audio(filename=str(path), normalize=False))
print("ไม่สรุปผล EER/accuracy ในขั้นนี้ และยังไม่เริ่ม train")